# Consistency Flow Matching：论文学习笔记

**论文名称：** *Consistency Flow Matching: Defining Straight Flows with Velocity Consistency*  
**论文链接：** https://arxiv.org/abs/2407.02398  
**研究方向：** Flow Matching、Consistency Models、Rectified Flow、少步生成  
**核心问题：** 如何通过速度一致性约束，使生成 ODE 的轨迹更加平直，从而减少采样所需的网络推理次数（NFE）。

> **笔记结构**
>
> 第一部分：论文总结，包括核心思想、主要结论、实验表现以及我们的理论质疑。
>
> 第二部分：数学基础，包括 Flow Matching 的条件期望、coupling、连续性方程，以及速度一致性与 Burgers 方程的联系。
>
> 第三部分：论文方法与推导，包括两个一致性条件、损失函数、Theorem 1、Theorem 2、分段训练和蒸馏。
>
> 第四部分：理论审查，分析速度一致性是否能够保证直线生成、正确的边际分布以及最优传输。

---

# 第一部分：论文总结与评价

## 1. 论文的核心思想

标准 Flow Matching 使用线性插值构造训练样本：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{1.1}
$$

其中：

- $X_0\sim p_0$：初始噪声；
- $X_1\sim p_1$：真实数据；
- $t\in[0,1]$：生成时间。

对应的条件速度：

$$
\boxed{
V=X_1-X_0
}
\tag{1.2}
$$

对于任意固定样本对：

$$
(X_0,X_1),
$$

速度 $V$ 不随时间变化，因此训练插值路径是一条匀速直线。

但是，Flow Matching 使用 MSE 训练：

$$
\mathcal L_{\mathrm{FM}}
=
\mathbb E
\left[
\|v_\theta(X_t,t)-V\|^2
\right].
$$

其最优解为：

$$
\boxed{
v^*(x,t)
=
\mathbb E[X_1-X_0\mid X_t=x]
}
\tag{1.3}
$$

即边缘速度场。

由于多个不同的样本对可能插值到同一个 $x$，而这些样本对对应的速度不同，因此模型学习到的是条件速度的加权平均。

所以：

$$
\boxed{
\text{训练插值路径是直线}
\not\Rightarrow
\text{生成 ODE 轨迹是直线}
}
\tag{1.4}
$$

这是论文试图解决的问题。

### Consistency-FM 的做法

论文提出在训练过程中额外约束速度一致性：

$$
\boxed{
v(x_t,t)=v(x_s,s)
}
\tag{1.5}
$$

这里 $x_t,x_s$ 应当是同一条生成 ODE 轨迹上的两个状态。

如果速度沿生成轨迹保持常数，那么：

$$
\frac{dx_t}{dt}=v_0.
$$

积分得到：

$$
\boxed{
x_t=x_0+tv_0
}
\tag{1.6}
$$

因此生成轨迹为直线。

为了实现这一目标，论文设计了两个一致性损失：

**终点一致性：**

$$
\boxed{
\mathcal L_{\mathrm{endpoint}}
=
\mathbb E
\left[
\|f_\theta(X_t,t)-f_{\theta^-}(X_{t+\Delta},t+\Delta)\|^2
\right]
}
\tag{1.7}
$$

**速度一致性：**

$$
\boxed{
\mathcal L_{\mathrm{velocity}}
=
\mathbb E
\left[
\|v_\theta(X_t,t)-v_{\theta^-}(X_{t+\Delta},t+\Delta)\|^2
\right]
}
\tag{1.8}
$$

其中：

$$
\boxed{
f_\theta(x,t)=x+(1-t)v_\theta(x,t)
}
\tag{1.9}
$$

表示从当前状态出发，假设预测速度保持不变，外推到 $t=1$ 得到的终点。

最终：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathcal L_{\mathrm{endpoint}}
+
\eta\mathcal L_{\mathrm{velocity}}
}
\tag{1.10}
$$

这里 $\eta$ 是速度一致性损失的权重。

论文进一步提出 Multi-Segment Consistency-FM，将整个生成区间划分为多个时间段，每个时间段分别约束速度一致性，以平衡生成轨迹的表达能力与采样效率。

## 2. 论文的主要理论结论

| 理论结果 | 论文证明或分析的内容 | 需要注意的限制 |
|---|---|---|
| Lemma 1 | 沿 ODE 轨迹的速度一致性与终点外推一致性等价 | 假设比较的是同一条 ODE 轨迹上的状态 |
| Lemma 2 | 速度一致性对应 $\partial_tv+J_vv=0$ | 这是速度场的动力学性质，不是分布匹配条件 |
| Theorem 1 | 终点一致性损失包含速度回归误差和沿真实流的速度变化项 | 两项存在交叉作用，不能直接视为两个独立平方损失 |
| Theorem 2 | 推导分段训练的速度预测误差递推关系 | 依赖局部最优化、目标速度和正则性等假设 |
| Corollary 2.1 | 当真实速度场本身一致时，可以恢复真实速度场 | 不保证一般弯曲概率流也能被正确直线化 |

论文最核心的数学条件为：

$$
\boxed{
\frac{d}{dt}v(x_t,t)=0
}
\tag{1.11}
$$

使用链式法则：

$$
\boxed{
\partial_tv(x,t)+J_v(x,t)v(x,t)=0
}
\tag{1.12}
$$

其中：

$$
J_v(x,t)=\frac{\partial v(x,t)}{\partial x}
$$

是速度场关于空间位置的 Jacobian。

这是多维无黏 Burgers 方程的形式。

**但必须强调：该方程描述的是沿生成轨迹速度保持不变，并不能单独保证生成分布正确。**

## 3. 论文的实验结论

论文主要在 CIFAR-10、AFHQ-Cat 和 CelebA-HQ 上进行图像生成实验。

报告的部分结果：

| 数据集 | 对比方法 | 对比方法 FID | Consistency-FM FID | NFE |
|---|---|---:|---:|---:|
| CIFAR-10 | Consistency Models | 5.83 | 5.34 | 2 |
| AFHQ-Cat | Rectified Flow | 61.5 | 22.5 | 6 |
| CelebA-HQ | Rectified Flow | 127.0 | 36.4 | 6 |

FID 越低越好。

作者还报告了相对于特定 Consistency Models 和 Rectified Flow 基线的训练收敛速度优势。

这些实验说明：

**Consistency-FM 在作者采用的实验配置下，能够改善少步生成质量。**

但不能仅凭 FID 改善证明：

- 模型生成 ODE 的轨迹严格变直；
- 模型保持了原始 Flow Matching 的边际概率路径；
- 模型学习到了最优 coupling；
- 该方法在相同计算预算下普遍优于其他少步生成方法。

因此，实验结果可以支持其作为一种有效训练策略，但不足以完全验证其理论解释。

---

# 第二部分：我们的核心质疑

这一部分先总结结论，后面再进行数学分析。

## 4. 质疑一：训练路径的一致性不等于生成路径的一致性

标准 Flow Matching 的训练状态为：

$$
X_t=(1-t)X_0+tX_1.
$$

因此：

$$
\frac{dX_t}{dt}=X_1-X_0.
$$

但模型生成时使用：

$$
\boxed{
\frac{dx_t}{dt}=v_\theta(x_t,t)
}
\tag{4.1}
$$

一般情况下：

$$
v_\theta(X_t,t)
\neq X_1-X_0.
$$

因此，训练样本对：

$$
(X_t,X_{t+\Delta})
$$

不一定对应模型自身 ODE 轨迹上的两个状态。

这意味着：

$$
v_\theta(X_t,t)
\approx
v_\theta(X_{t+\Delta},t+\Delta)
$$

并不直接推出：

$$
\boxed{
\frac{d}{dt}v_\theta(x_t,t)=0
}
$$

其中 $x_t$ 是模型自身生成的 ODE 轨迹。

**论文从训练路径一致性到生成 ODE 直线性的论证，需要额外假设。**

## 5. 质疑二：没有显式优化 coupling

对于固定 coupling：

$$
\pi(X_0,X_1),
$$

标准 Flow Matching 的最优速度为：

$$
\boxed{
u(x,t)
=
\mathbb E[X_1-X_0\mid X_t=x].
}
\tag{5.1}
$$

该速度场通常不满足：

$$
\partial_tu+J_uu=0.
$$

Consistency-FM 并没有显式改变训练样本的 coupling，而是修改了速度场的优化目标。

因此存在一个重要问题：

如果原始边缘速度场本身不一致，那么模型究竟是在逼近原始边缘速度场，还是通过牺牲原始概率路径的正确性来获得更强的速度一致性？

论文没有给出一般情况下的完整答案。

需要注意：不显式优化 coupling 并不意味着绝对不可能学习到新的直线输运；但要保证新的输运正确，需要额外的分布匹配条件。

## 6. 质疑三：速度一致性不能保证连续性方程

原始 Flow Matching 的边缘速度 $u$ 满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tu)=0.
}
\tag{6.1}
$$

如果 Consistency-FM 学到：

$$
v_\theta=u+\delta v,
$$

要保持原始概率路径 $p_t$，必须满足：

$$
\boxed{
\nabla\cdot(p_t\delta v)=0.
}
\tag{6.2}
$$

但速度一致性损失没有直接约束这个条件。

因此：

$$
\boxed{
\text{速度一致性}
\not\Rightarrow
\text{原始边际分布保持}
}
\tag{6.3}
$$

还要注意：即使原始中间边际 $p_t$ 不再保持，模型仍可能通过另一条概率路径正确到达 $p_1$。但这也需要单独证明。

## 7. 质疑四：直线输运不等于最优传输

最优传输需要优化 coupling：

$$
\boxed{
\min_{\pi\in\Pi(p_0,p_1)}
\mathbb E_\pi\|X_1-X_0\|^2.
}
\tag{7.1}
$$

而 Consistency-FM 主要约束：

$$
\boxed{
\partial_tv+J_vv=0.
}
\tag{7.2}
$$

前者是输运成本最小化问题。

后者是速度场的动力学一致性条件。

两者并不等价。

因此，即使 Consistency-FM 学到了直线生成轨迹，也不能据此证明其对应最优传输。

## 8. 质疑五：分段训练可能只是改善局部积分精度

Multi-Segment Consistency-FM 将：

$$
[0,1]
$$

划分为 $K$ 段，每段使用近似恒定的速度。

推理形式仍然是：

$$
\boxed{
x_{t_{i+1}}
=
x_{t_i}
+
\Delta t\,v_\theta^i(x_{t_i},t_i).
}
\tag{8.1}
$$

也就是 Euler 更新。

分段训练可能改善局部速度场的可积分性，但不能自动消除速度场本身的分布匹配误差。

因此：

$$
\boxed{
\text{积分误差减小}
\not\Rightarrow
\text{生成分布误差减小}
}
\tag{8.2}
$$

当然，实际生成质量可能同时受两种误差影响，因此分段方法仍然具有实验价值。

### 阶段性评价

综合来看，这篇论文可以被理解为：

**通过速度一致性正则化改善少步采样的训练方法。**

但不能将其理解为已经从理论上解决了 Flow Matching 的 coupling 问题、一般直线输运问题或最优传输问题。

下面进入论文的数学基础与方法推导。

---

# 第三部分：数学基础与核心推导

## 9. Flow Matching 为什么会产生弯曲的生成轨迹？

这一节是理解论文的基础，也是判断 Consistency-FM 是否合理的关键。

### 9.1 首先区分三个不同的对象

设：

$$
(X_0,X_1)\sim\pi.
$$

其中：

$$
\pi
$$

是噪声分布与数据分布之间的 coupling。

我们需要区分：

**第一个对象：条件插值路径**

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{9.1}
$$

对于给定的一对：

$$
(X_0,X_1),
$$

它是一条确定的直线。

**第二个对象：条件速度随机变量**

$$
\boxed{
V=X_1-X_0
}
\tag{9.2}
$$

对于同一对样本，$V$ 不随时间变化。

**第三个对象：边缘速度场**

$$
\boxed{
u(x,t)=\mathbb E[V\mid X_t=x].
}
\tag{9.3}
$$

它是模型使用 MSE 训练后真正趋向学习的速度场。

生成时积分的是：

$$
\boxed{
\frac{dx_t}{dt}=u(x_t,t),
}
\tag{9.4}
$$

而不是对某一对样本直接积分：

$$
\frac{dX_t}{dt}=X_1-X_0.
$$

因此，不能把条件路径的性质直接赋予边缘 ODE 轨迹。

### 9.2 从 MSE 推导条件期望

标准 Flow Matching：

$$
\boxed{
\mathcal L_{\mathrm{FM}}
=
\mathbb E\|v_\theta(X_t,t)-V\|^2.
}
\tag{9.5}
$$

固定：

$$
X_t=x,
$$

模型只能输出一个向量：

$$
a=v_\theta(x,t).
$$

于是：

$$
\min_a\mathbb E[\|a-V\|^2\mid X_t=x].
$$

展开：

$$
\begin{aligned}
\mathbb E[\|a-V\|^2\mid X_t=x]
={}&
a^\top a\\
&-2a^\top\mathbb E[V\mid X_t=x]\\
&+\mathbb E[V^\top V\mid X_t=x].
\end{aligned}
$$

对 $a$ 求导：

$$
2a-2\mathbb E[V\mid X_t=x]=0.
$$

因此：

$$
\boxed{
v^*(x,t)=\mathbb E[V\mid X_t=x].
}
\tag{9.6}
$$

对于直线插值：

$$
V=X_1-X_0,
$$

所以：

$$
\boxed{
v^*(x,t)
=
\mathbb E[X_1-X_0\mid X_t=x].
}
\tag{9.7}
$$

这就是条件速度的加权平均。

### 9.3 为什么一个 $X_t=x$ 可以对应多个样本对？

由：

$$
x=(1-t)x_0+tx_1,
$$

对于：

$$
0<t<1,
$$

如果固定 $x_0$，那么：

$$
\boxed{
x_1=\frac{x-(1-t)x_0}{t}.
}
\tag{9.8}
$$

因此，不同的 $x_0$ 可以对应不同的 $x_1$，并且都满足：

$$
X_t=x.
$$

它们的速度：

$$
V=x_1-x_0
$$

一般不同。

将式（9.8）代入：

$$
V=\frac{x-x_0}{t}.
$$

所以：

$$
\boxed{
u(x,t)
=
\frac{x-\mathbb E[X_0\mid X_t=x]}{t}.
}
\tag{9.9}
$$

同理：

$$
\boxed{
u(x,t)
=
\frac{\mathbb E[X_1\mid X_t=x]-x}{1-t}.
}
\tag{9.10}
$$

这里的条件期望是由 coupling $\pi$ 决定的。

因此：

$$
\boxed{
\pi
\longrightarrow
p_t
\longrightarrow
u(x,t).
}
\tag{9.11}
$$

改变 coupling，就可能改变边缘速度场。

### 9.4 一个解析反例：训练路径是直线，但边缘 ODE 弯曲

考虑：

$$
X_0,X_1\overset{\mathrm{i.i.d.}}{\sim}\mathcal N(0,1).
$$

使用独立 coupling：

$$
\pi=p_0p_1.
$$

线性插值：

$$
X_t=(1-t)X_0+tX_1.
$$

由于两个高斯变量独立：

$$
X_t\sim\mathcal N(0,D(t)),
$$

其中：

$$
\boxed{
D(t)=(1-t)^2+t^2.
}
\tag{9.12}
$$

条件速度：

$$
V=X_1-X_0.
$$

因为 $(V,X_t)$ 联合高斯：

$$
\mathbb E[V\mid X_t=x]
=
\frac{\operatorname{Cov}(V,X_t)}
{\operatorname{Var}(X_t)}x.
$$

计算：

$$
\begin{aligned}
\operatorname{Cov}(V,X_t)
&=
\operatorname{Cov}
(X_1-X_0,(1-t)X_0+tX_1)\\
&=t-(1-t)\\
&=2t-1.
\end{aligned}
$$

所以：

$$
\boxed{
u(x,t)=\frac{2t-1}{D(t)}x.
}
\tag{9.13}
$$

生成 ODE：

$$
\frac{dx_t}{dt}
=
\frac{2t-1}{D(t)}x_t.
$$

注意：

$$
D'(t)=2(2t-1).
$$

因此：

$$
\frac{d}{dt}\ln|x_t|
=
\frac12\frac{D'(t)}{D(t)}.
$$

积分：

$$
\ln|x_t|
=
\frac12\ln D(t)+C.
$$

使用初始条件：

$$
D(0)=1,
$$

得到：

$$
\boxed{
x_t=\sqrt{D(t)}\,x_0.
}
\tag{9.14}
$$

显然：

$$
x_t
$$

不是时间的线性函数。

例如：

$$
x_{1/2}=\frac{x_0}{\sqrt2}.
$$

而：

$$
x_1=x_0.
$$

也就是说，边缘 ODE 先将样本向原点收缩，再恢复到原始位置。

虽然每一对条件插值样本都是匀速直线，但最终边缘 ODE 不是匀速直线。

如果改成：

$$
X_1=X_0,
$$

那么：

$$
V=0,
$$

模型学习到：

$$
u(x,t)=0.
$$

此时生成 ODE：

$$
x_t=x_0
$$

就是一条静止的直线轨迹。

两种情况的初始分布和目标分布完全相同。

唯一改变的是 coupling。

因此：

$$
\boxed{
\text{边缘 ODE 的直线性与 coupling 密切相关。}
}
\tag{9.15}
$$

但这并不意味着任意改变 coupling 都能得到合法的直线 ODE；还需要考虑轨迹交叉、速度场正则性以及流映射可逆性。

---

## 10. Flow Matching 为什么能够保证边际分布？

论文的一个关键前提是：真实边缘速度场满足连续性方程。

这里不能只引用结论，需要从条件插值推导。

### 10.1 从随机插值出发

设：

$$
X_t=(1-t)X_0+tX_1.
$$

其概率密度为：

$$
p_t(x).
$$

对于任意足够光滑的测试函数：

$$
\varphi:\mathbb R^d\rightarrow\mathbb R,
$$

有：

$$
\mathbb E[\varphi(X_t)]
=
\int\varphi(x)p_t(x)\,dx.
$$

对时间求导：

$$
\frac{d}{dt}\mathbb E[\varphi(X_t)]
=
\mathbb E\left[
\nabla\varphi(X_t)^\top
\frac{dX_t}{dt}
\right].
$$

由于：

$$
\frac{dX_t}{dt}=V,
$$

得到：

$$
\frac{d}{dt}\mathbb E[\varphi(X_t)]
=
\mathbb E[\nabla\varphi(X_t)^\top V].
$$

使用条件期望：

$$
\begin{aligned}
\mathbb E[\nabla\varphi(X_t)^\top V]
&=
\mathbb E\left[
\nabla\varphi(X_t)^\top
\mathbb E[V\mid X_t]
\right]\\
&=
\int
\nabla\varphi(x)^\top
u(x,t)p_t(x)\,dx.
\end{aligned}
$$

另一方面：

$$
\frac{d}{dt}\mathbb E[\varphi(X_t)]
=
\int\varphi(x)\partial_tp_t(x)\,dx.
$$

因此：

$$
\int\varphi(x)\partial_tp_t(x)\,dx
=
\int\nabla\varphi(x)^\top u(x,t)p_t(x)\,dx.
$$

对右侧分部积分：

$$
\int\nabla\varphi^\top up_t\,dx
=
-\int\varphi\,\nabla\cdot(up_t)\,dx.
$$

所以：

$$
\int\varphi(x)
\left[
\partial_tp_t+\nabla\cdot(p_tu)
\right]dx=0.
$$

由于对任意测试函数成立：

$$
\boxed{
\partial_tp_t(x)
+
\nabla\cdot[p_t(x)u(x,t)]
=
0.
}
\tag{10.1}
$$

这就是连续性方程。

### 10.2 连续性方程的物理意义

它表达的是概率质量守恒。

对于空间中的一个区域：

$$
\Omega,
$$

区域内的概率质量：

$$
P_t(\Omega)=\int_\Omega p_t(x)\,dx.
$$

对时间求导：

$$
\frac{d}{dt}P_t(\Omega)
=
\int_\Omega\partial_tp_t(x)\,dx.
$$

代入连续性方程：

$$
=
-\int_\Omega\nabla\cdot(p_tu)\,dx.
$$

根据散度定理：

$$
\boxed{
\frac{d}{dt}P_t(\Omega)
=
-\int_{\partial\Omega}
p_tu\cdot n\,dS.
}
\tag{10.2}
$$

其中：

- $\partial\Omega$：区域边界；
- $n$：向外的单位法向量；
- $dS$：边界面积元素。

这表示：

> 区域内概率质量的变化，等于穿过边界的净概率流量的负值。

因此，连续性方程保证了概率质量按照速度场运动。

在适当正则性和解唯一性条件下，如果：

$$
v_\theta=u,
$$

那么从：

$$
x_0\sim p_0
$$

出发积分：

$$
\frac{dx_t}{dt}=v_\theta(x_t,t)
$$

就能够生成目标概率路径：

$$
p_t.
$$

这就是标准 Flow Matching 的理论基础。

---

## 11. 速度一致性为什么对应 Burgers 方程？

论文 Lemma 2 给出：

$$
\boxed{
\partial_tv+v\cdot\nabla_xv=0.
}
\tag{11.1}
$$

这不是作者新提出的方程，而是经典的无黏 Burgers 方程。

我们从粒子运动推导，而不是直接引用名称。

### 11.1 从速度场定义粒子运动

考虑一维空间中的粒子：

$$
x_t\in\mathbb R.
$$

速度场：

$$
v(x,t)
$$

表示时间 $t$、位置 $x$ 处的速度。

粒子运动满足：

$$
\boxed{
\frac{dx_t}{dt}=v(x_t,t).
}
\tag{11.2}
$$

假设每个粒子在运动过程中速度保持不变：

$$
\boxed{
\frac{d}{dt}v(x_t,t)=0.
}
\tag{11.3}
$$

注意：

$$
v(x_t,t)
$$

是复合函数，因为位置 $x_t$ 本身依赖时间。

使用链式法则：

$$
\frac{d}{dt}v(x_t,t)
=
\frac{\partial v}{\partial t}
+
\frac{\partial v}{\partial x}
\frac{dx_t}{dt}.
$$

代入：

$$
\frac{dx_t}{dt}=v(x_t,t),
$$

得到：

$$
\frac{d}{dt}v(x_t,t)
=
\partial_tv+v\partial_xv.
$$

因此：

$$
\boxed{
\partial_tv+v\partial_xv=0.
}
\tag{11.4}
$$

这就是一维无黏 Burgers 方程。

对于多维速度场：

$$
v(x,t)\in\mathbb R^d,
$$

同理得到：

$$
\boxed{
\partial_tv+J_vv=0.
}
\tag{11.5}
$$

其中：

$$
J_v=\frac{\partial v}{\partial x}
\in\mathbb R^{d\times d}.
$$

### 11.2 特征线是什么？

特征线法是求解一阶偏微分方程的一种方法。

其核心思想是：

**寻找一条空间运动轨迹，使偏微分方程沿该轨迹变成常微分方程。**

对于 Burgers 方程：

$$
\partial_tv+v\partial_xv=0,
$$

选择轨迹：

$$
\boxed{
\frac{dx_t}{dt}=v(x_t,t).
}
\tag{11.6}
$$

沿这条轨迹：

$$
\frac{d}{dt}v(x_t,t)
=
\partial_tv+v\partial_xv.
$$

根据 Burgers 方程：

$$
\frac{d}{dt}v(x_t,t)=0.
$$

因此：

$$
v(x_t,t)=v(x_0,0).
$$

定义：

$$
v_0=v(x_0,0).
$$

那么：

$$
\frac{dx_t}{dt}=v_0.
$$

积分：

$$
\boxed{
x_t=x_0+tv_0.
}
\tag{11.7}
$$

这就是特征线。

所以这里的特征线并不是另一种新的生成轨迹，而是速度场对应的 ODE 运动轨迹。

我们不需要额外引入：

$$
\gamma_x(t).
$$

直接使用：

$$
x_t
$$

即可。

### 11.3 为什么这个方程会限制 coupling？

设一维初始速度为：

$$
v_0(x).
$$

从初始位置 $a$ 出发：

$$
x_t(a)=a+tv_0(a).
$$

从位置 $b$ 出发：

$$
x_t(b)=b+tv_0(b).
$$

如果存在某个时间：

$$
t^*
$$

满足：

$$
x_{t^*}(a)=x_{t^*}(b),
$$

但：

$$
v_0(a)\neq v_0(b),
$$

那么同一时间、同一位置要求速度场同时具有两个不同的速度。

这与单值速度场矛盾。

因此，对于光滑、单值的 ODE 速度场，不能任意指定相互交叉且速度不同的直线轨迹。

这说明：

$$
\boxed{
\text{全局直线 ODE 对输运映射和 coupling 有结构性要求。}
}
\tag{11.8}
$$

这也是为什么仅仅约束速度一致性，并不能保证任意 coupling 都可以被准确表示。

---

# 第四部分：Consistency-FM 方法与理论推导

## 12. Lemma 1：速度一致性与终点一致性等价

论文首先定义生成 ODE：

$$
\boxed{
\frac{dx_t}{dt}=v(x_t,t).
}
\tag{12.1}
$$

其中：

$$
x_0
$$

是初始状态。

论文提出两个条件。

### Condition 1：速度一致性

$$
\boxed{
v(x_t,t)=v(x_s,s),
\qquad \forall s,t\in[0,1].
}
\tag{12.2}
$$

也就是说，沿同一条 ODE 轨迹，速度不随时间变化。

### Condition 2：终点外推一致性

$$
\boxed{
x_t+(1-t)v(x_t,t)
=
x_s+(1-s)v(x_s,s).
}
\tag{12.3}
$$

它表示：

从同一条轨迹的不同时间点出发，按照当前速度直线外推到 $t=1$，得到的终点相同。

论文在速度场关于空间满足 Lipschitz 条件、关于时间和空间可微等假设下，证明这两个条件等价。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

### 12.1 Condition 1 推导 Condition 2

如果速度一致：

$$
v(x_t,t)=v_0,
$$

那么：

$$
\frac{dx_t}{dt}=v_0.
$$

积分：

$$
x_t=x_0+tv_0.
$$

因此：

$$
\begin{aligned}
x_t+(1-t)v(x_t,t)
&=x_0+tv_0+(1-t)v_0\\
&=x_0+v_0.
\end{aligned}
$$

所以：

$$
\boxed{
x_t+(1-t)v(x_t,t)=x_1.
}
\tag{12.4}
$$

右侧与时间无关。

因此 Condition 2 成立。

### 12.2 Condition 2 推导 Condition 1

定义：

$$
\boxed{
F(t)=x_t+(1-t)v(x_t,t).
}
\tag{12.5}
$$

Condition 2 表示：

$$
F(t)=C,
$$

其中 $C$ 是常数。

因此：

$$
\frac{dF(t)}{dt}=0.
$$

对式（12.5）求导：

$$
\begin{aligned}
\frac{dF}{dt}
={}&
\frac{dx_t}{dt}
-v(x_t,t)\\
&+
(1-t)\frac{d}{dt}v(x_t,t).
\end{aligned}
$$

根据 ODE：

$$
\frac{dx_t}{dt}=v(x_t,t),
$$

前两项抵消：

$$
\boxed{
\frac{dF}{dt}
=
(1-t)\frac{d}{dt}v(x_t,t).
}
\tag{12.6}
$$

由于：

$$
\frac{dF}{dt}=0,
$$

所以：

$$
(1-t)\frac{d}{dt}v(x_t,t)=0.
$$

当：

$$
t<1,
$$

得到：

$$
\boxed{
\frac{d}{dt}v(x_t,t)=0.
}
\tag{12.7}
$$

在连续性假设下可延伸至端点。

因此 Condition 1 成立。

最终：

$$
\boxed{
\text{Condition 1}
\Longleftrightarrow
\text{Condition 2}.
}
\tag{12.8}
$$

### 12.3 这个定理证明了什么，没有证明什么？

它证明：

如果两个条件在**同一条模型 ODE 轨迹**上成立，那么模型生成轨迹是直线。

但论文训练时使用的：

$$
X_t,\quad X_{t+\Delta}
$$

来自预定义概率路径，而不一定来自模型自身 ODE。

因此，这个定理不能直接保证训练损失下降后，模型生成轨迹就严格变直。

这就是我们前面提出的第一个理论质疑。

---

## 13. Consistency-FM 的训练目标

论文并没有直接最小化 Burgers 方程残差：

$$
\|\partial_tv_\theta+J_{v_\theta}v_\theta\|^2.
$$

而是采用相邻时间点的一致性训练。

### 13.1 定义终点预测函数

$$
\boxed{
f_\theta(x,t)
=
x+(1-t)v_\theta(x,t).
}
\tag{13.1}
$$

其中：

- $x$：当前生成状态；
- $t$：当前生成时间；
- $v_\theta(x,t)$：预测速度；
- $f_\theta(x,t)$：沿当前速度外推到终点得到的位置。

注意：

$$
f_\theta
$$

不是一个独立网络，而是由速度模型构造。

### 13.2 构造相邻训练状态

采样：

$$
t\sim\mathcal U(0,1-\Delta).
$$

定义：

$$
t'=t+\Delta.
$$

例如使用线性 OT 路径：

$$
X_t=(1-t)X_0+tX_1,
$$

$$
X_{t'}=(1-t')X_0+t'X_1.
$$

注意两个状态使用同一对：

$$
(X_0,X_1).
$$

因此：

$$
\boxed{
X_{t'}-X_t=\Delta(X_1-X_0).
}
\tag{13.2}
$$

### 13.3 在线模型与 EMA 模型

论文使用在线模型：

$$
v_\theta
$$

和 EMA 目标模型：

$$
v_{\theta^-}.
$$

EMA 更新：

$$
\boxed{
\theta^-
\leftarrow
\beta\theta^-+(1-\beta)\theta.
}
\tag{13.3}
$$

目标分支停止梯度传播，用于稳定训练。

### 13.4 终点一致性损失

在线模型：

$$
f_\theta(X_t,t)
=
X_t+(1-t)v_\theta(X_t,t).
$$

EMA 模型：

$$
f_{\theta^-}(X_{t'},t')
=
X_{t'}+(1-t')v_{\theta^-}(X_{t'},t').
$$

要求：

$$
\boxed{
\mathcal L_{\mathrm{endpoint}}
=
\mathbb E
\left[
\|f_\theta(X_t,t)-f_{\theta^-}(X_{t'},t')\|^2
\right].
}
\tag{13.4}
$$

### 13.5 速度一致性损失

$$
\boxed{
\mathcal L_{\mathrm{velocity}}
=
\mathbb E
\left[
\|v_\theta(X_t,t)-v_{\theta^-}(X_{t'},t')\|^2
\right].
}
\tag{13.5}
$$

最终：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}
=
\mathcal L_{\mathrm{endpoint}}
+
\eta\mathcal L_{\mathrm{velocity}}.
}
\tag{13.6}
$$

这对应论文公式（6）。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

### 13.6 两个 Loss 是否重复？

定义：

$$
v_t=v_\theta(X_t,t),
$$

$$
v_{t'}=v_{\theta^-}(X_{t'},t').
$$

终点残差：

$$
\begin{aligned}
R_f
&=
f_\theta(X_t,t)-f_{\theta^-}(X_{t'},t')\\
&=
X_t-X_{t'}+(1-t)v_t-(1-t')v_{t'}.
\end{aligned}
$$

因为：

$$
t'=t+\Delta,
$$

所以：

$$
1-t'=1-t-\Delta.
$$

代入：

$$
R_f
=
X_t-X_{t'}+(1-t)v_t-(1-t-\Delta)v_{t'}.
$$

整理：

$$
\begin{aligned}
R_f
={}&
X_t-X_{t'}\\
&+
\Delta v_t\\
&+
(1-t-\Delta)(v_t-v_{t'}).
\end{aligned}
$$

利用：

$$
X_{t'}-X_t=\Delta(X_1-X_0),
$$

得到：

$$
\boxed{
\begin{aligned}
R_f
={}&
\Delta[v_t-(X_1-X_0)]\\
&+
(1-t-\Delta)(v_t-v_{t'}).
\end{aligned}
}
\tag{13.7}
$$

这说明终点一致性残差包含：

1. 对条件速度的回归误差；
2. 相邻时间点的速度一致性误差。

但平方之后存在交叉项：

$$
\|A+B\|^2
=
\|A\|^2+\|B\|^2+2A^\top B.
$$

因此，终点一致性损失不等于两个独立平方损失的简单相加。

而且两个误差可能相互抵消。

这也是额外引入速度一致性损失的一个合理动机。

但需要注意：

**增加速度一致性损失并不能自动保证模型恢复原始边缘速度场。**

---

## 14. Theorem 1：一致性损失与真实边缘速度之间有什么关系？

这是论文最重要的理论分析，对应原文公式（7）。

作者希望说明，终点一致性损失不仅约束终点预测，还与 Flow Matching 的速度回归误差有关。

不过，要理解这个结论，需要先明确一个细节：

**原始训练中的两个状态来自预定义概率路径，而理论展开使用的是该概率路径对应的真实边缘速度场。**

下面先按照论文的理论设定推导，再讨论这个区别。

### 14.1 定义真实边缘速度场

设：

$$
u(x,t)
$$

是能够生成目标概率路径 $p_t$ 的真实边缘速度场。

它满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tu)=0.
}
\tag{14.1}
$$

假设我们沿真实边缘 ODE 运动：

$$
\boxed{
\frac{dx_t}{dt}=u(x_t,t).
}
\tag{14.2}
$$

对于一个较小的时间间隔：

$$
\Delta>0,
$$

有：

$$
\boxed{
x_{t+\Delta}
=
x_t+\Delta u(x_t,t)+O(\Delta^2).
}
\tag{14.3}
$$

其中：

$$
O(\Delta^2)
$$

表示二阶及以上的局部误差。

### 14.2 对模型速度进行 Taylor 展开

我们需要展开：

$$
v_\theta(x_{t+\Delta},t+\Delta).
$$

由于时间和位置都发生变化，使用多元 Taylor 展开：

$$
\begin{aligned}
v_\theta(x_{t+\Delta},t+\Delta)
={}&
v_\theta(x_t,t)\\
&+
\Delta\partial_tv_\theta(x_t,t)\\
&+
J_{v_\theta}(x_t,t)
(x_{t+\Delta}-x_t)\\
&+
O(\Delta^2).
\end{aligned}
$$

代入：

$$
x_{t+\Delta}-x_t
=
\Delta u(x_t,t)+O(\Delta^2),
$$

得到：

$$
\boxed{
\begin{aligned}
v_\theta(x_{t+\Delta},t+\Delta)
={}&
v_\theta(x_t,t)\\
&+
\Delta
\left[
\partial_tv_\theta+
J_{v_\theta}u
\right]\\
&+
O(\Delta^2).
\end{aligned}
}
\tag{14.4}
$$

定义：

$$
\boxed{
D_t^uv_\theta
=
\partial_tv_\theta+J_{v_\theta}u.
}
\tag{14.5}
$$

它表示沿真实边缘速度场 $u$ 的方向导数。

这里要特别区分：

$$
D_t^uv_\theta
=
\partial_tv_\theta+J_{v_\theta}u
$$

和：

$$
D_t^{v_\theta}v_\theta
=
\partial_tv_\theta+J_{v_\theta}v_\theta.
$$

前者沿真实概率流求导。

后者沿模型自身 ODE 求导。

**只有当 $v_\theta=u$ 时，两者才相同。**

因此，Theorem 1 中出现的方向导数，并不是严格意义上模型自身 ODE 的速度一致性残差。

### 14.3 展开终点一致性残差

定义：

$$
f_\theta(x,t)=x+(1-t)v_\theta(x,t).
$$

考虑：

$$
R_f
=
f_\theta(x_t,t)
-
f_\theta(x_{t+\Delta},t+\Delta).
$$

展开：

$$
\begin{aligned}
R_f
={}&
x_t-x_{t+\Delta}\\
&+
(1-t)v_\theta(x_t,t)\\
&-
(1-t-\Delta)v_\theta(x_{t+\Delta},t+\Delta).
\end{aligned}
$$

由式（14.3）：

$$
x_t-x_{t+\Delta}
=
-\Delta u+O(\Delta^2).
$$

由式（14.4）：

$$
v_\theta(x_{t+\Delta},t+\Delta)
=
v_\theta+\Delta D_t^uv_\theta+O(\Delta^2).
$$

代入：

$$
\begin{aligned}
R_f
={}&
-\Delta u\\
&+
(1-t)v_\theta\\
&-
(1-t-\Delta)
(v_\theta+\Delta D_t^uv_\theta)\\
&+
O(\Delta^2).
\end{aligned}
$$

展开乘积：

$$
\begin{aligned}
R_f
={}&
-\Delta u\\
&+
(1-t)v_\theta\\
&-
(1-t)v_\theta\\
&+
\Delta v_\theta\\
&-
(1-t)\Delta D_t^uv_\theta\\
&+
O(\Delta^2).
\end{aligned}
$$

抵消相同项：

$$
\boxed{
R_f
=
\Delta
\left[
v_\theta-u-(1-t)D_t^uv_\theta
\right]
+
O(\Delta^2).
}
\tag{14.6}
$$

代入方向导数定义：

$$
\boxed{
\begin{aligned}
R_f
=
\Delta\Big[
&
v_\theta-u\\
&-
(1-t)
(\partial_tv_\theta+J_{v_\theta}u)
\Big]
+
O(\Delta^2).
\end{aligned}
}
\tag{14.7}
$$

### 14.4 对残差取平方

根据：

$$
R_f=\Delta A+O(\Delta^2),
$$

有：

$$
\|R_f\|^2
=
\Delta^2\|A\|^2+o(\Delta^2).
$$

因此：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{endpoint}}
=
\Delta^2\mathbb E\Big\|
&
v_\theta-u\\
&-
(1-t)
(\partial_tv_\theta+J_{v_\theta}u)
\Big\|^2\\
&+
o(\Delta^2).
\end{aligned}
}
\tag{14.8}
$$

这对应论文 Theorem 1。

论文假设 $v_\theta$ 有界、二阶连续可微且相关导数有界，并假设真实速度场满足相应的有界性条件。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

### 14.5 这里存在一个需要单独指出的数学细节

刚才的 Taylor 展开使用：

$$
x_{t+\Delta}
=
x_t+\Delta u(x_t,t)+O(\Delta^2).
$$

这对于真实边缘 ODE 轨迹成立。

但对于线性条件插值：

$$
X_t=(1-t)X_0+tX_1,
$$

我们实际有：

$$
\boxed{
X_{t+\Delta}
=
X_t+\Delta V,
}
\tag{14.9}
$$

其中：

$$
V=X_1-X_0.
$$

一般来说：

$$
V\neq u(X_t,t).
$$

所以不能直接把式（14.9）中的 $V$ 替换为 $u$，然后认为逐样本 Taylor 展开完全相同。

更严格地说，沿条件插值路径展开时：

$$
\begin{aligned}
R_f
=
\Delta\Big[
&
v_\theta-V\\
&-
(1-t)
(\partial_tv_\theta+J_{v_\theta}V)
\Big]
+
O(\Delta^2).
\end{aligned}
\tag{14.10}
$$

现在定义：

$$
A(x,t)
=
v_\theta-u-(1-t)(\partial_tv_\theta+J_{v_\theta}u).
$$

以及：

$$
M(x,t)
=
I-(1-t)J_{v_\theta}(x,t).
$$

则：

$$
\begin{aligned}
&v_\theta-V-(1-t)(\partial_tv_\theta+J_{v_\theta}V)\\
={}&
A(x,t)+M(x,t)(u-V).
\end{aligned}
$$

因此：

$$
\boxed{
\frac{\mathcal L_{\mathrm{endpoint}}}{\Delta^2}
=
\mathbb E\|A+M(u-V)\|^2+o(1).
}
\tag{14.11}
$$

由于：

$$
\mathbb E[u-V\mid X_t=x]=0,
$$

且 $A,M$ 在给定 $X_t=x$ 时是确定的，交叉项的条件期望为零。

于是：

$$
\boxed{
\begin{aligned}
\frac{\mathcal L_{\mathrm{endpoint}}}{\Delta^2}
={}&
\mathbb E\|A(X_t,t)\|^2\\
&+
\mathbb E\left[
\operatorname{tr}
\left(
M\Sigma_{V\mid X_t}M^\top
\right)
\right]\\
&+
o(1),
\end{aligned}
}
\tag{14.12}
$$

其中：

$$
\boxed{
\Sigma_{V\mid X_t}
=
\operatorname{Cov}(V\mid X_t,t).
}
\tag{14.13}
$$

这个额外的条件方差项非常重要。

它说明：**如果相邻训练状态来自同一对独立采样的噪声—数据条件直线，那么一般不能直接把训练损失等同于沿真实边缘 ODE 的损失。**

只有在额外条件下，例如：

$$
\operatorname{Cov}(V\mid X_t,t)=0,
$$

或者使用真实边缘 ODE 耦合来构造相邻状态，才可以消除相应差异。

这是我们基于论文目标进行的进一步推导，并非论文明确给出的 Theorem 1 结论。

它也使我们之前对训练路径和生成路径不一致的质疑更加具体。

### 14.6 Theorem 1 的结论应该怎样理解？

论文希望说明：

$$
\mathcal L_{\mathrm{endpoint}}
$$

同时涉及：

$$
v_\theta-u
$$

和：

$$
\partial_tv_\theta+J_{v_\theta}u.
$$

但不能由此推出：

$$
v_\theta=u
$$

以及：

$$
\partial_tv_\theta+J_{v_\theta}v_\theta=0
$$

会同时成立。

原因有三点：

第一，两个误差在同一个平方范数中，可能相互抵消。

第二，沿真实速度场的方向导数，不等于沿模型速度场的方向导数。

第三，如果训练使用条件直线耦合，还需要处理条件速度方差。

因此：

$$
\boxed{
\text{Theorem 1 是损失结构分析，不是一般性的正确直线输运保证。}
}
\tag{14.14}
$$

---

## 15. Multi-Segment Consistency-FM：分段一致性

论文发现，要求整个时间区间内速度保持一致，可能限制模型表达能力。

因此提出将时间划分为多个区间。

### 15.1 时间分段

将：

$$
[0,1]
$$

划分为 $K$ 段。

定义：

$$
\boxed{
a_i=\frac{i}{K},
\qquad
b_i=\frac{i+1}{K},
}
\tag{15.1}
$$

其中：

$$
i=0,1,\ldots,K-1.
$$

每段区间：

$$
t\in[a_i,b_i].
$$

在第 $i$ 段学习速度场：

$$
v_\theta^i(x,t).
$$

要求它在当前分段内近似一致。

### 15.2 分段终点预测

单段方法的终点为：

$$
t=1.
$$

分段方法中，第 $i$ 段的终点为：

$$
b_i.
$$

因此定义：

$$
\boxed{
f_\theta^i(x,t)
=
x+(b_i-t)v_\theta^i(x,t).
}
\tag{15.2}
$$

它表示从当前状态出发，沿当前速度外推到该分段终点。

### 15.3 分段损失

论文公式（8）：

$$
\boxed{
\begin{aligned}
\mathcal L_i
=
\mathbb E\Big[
&
\omega_i
\|f_\theta^i(X_t,t)
-
f_{\theta^-}^i(X_{t+\Delta},t+\Delta)\|^2\\
&+
\eta
\|v_\theta^i(X_t,t)
-
v_{\theta^-}^i(X_{t+\Delta},t+\Delta)\|^2
\Big].
\end{aligned}
}
\tag{15.3}
$$

其中：

- $\omega_i>0$：第 $i$ 段终点一致性损失权重；
- $\eta>0$：速度一致性权重；
- $t\sim\mathcal U(a_i,b_i-\Delta)$。

论文原文使用 $\lambda^i$ 和 $\alpha$ 表示这两个权重。这里统一改写符号，避免与其他扩散模型参数混淆。

需要注意，$\omega_i$ 只作用于终点一致性项，后面的速度一致性项并没有乘以 $\omega_i$。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

---

## 16. Theorem 2：分段速度预测误差的推导

论文希望分析：

> 当真实速度场并不完全一致时，分段 Consistency-FM 的预测误差由哪些部分组成？

这一部分对应论文公式（9）、（10）和附录 A.4。

### 16.1 定义当前分段

考虑时间区间：

$$
[S,T].
$$

这里：

$$
T
$$

是当前分段的终点。

定义：

$$
A=T-t,
$$

$$
B=T-t-\Delta.
$$

显然：

$$
B=A-\Delta.
$$

记：

$$
v_t=v_\theta(x_t,t),
$$

$$
v_+=v_{\theta^-}(x_{t+\Delta},t+\Delta).
$$

暂时将：

$$
v_+
$$

视为固定目标。

### 16.2 写出局部损失

当前时刻的终点预测：

$$
f_t=x_t+Av_t.
$$

下一时刻的终点预测：

$$
f_+=x_{t+\Delta}+Bv_+.
$$

忽略分段权重，先考虑：

$$
\boxed{
\mathcal L(v_t)
=
\|x_t+Av_t-x_{t+\Delta}-Bv_+\|^2
+
\eta\|v_t-v_+\|^2.
}
\tag{16.1}
$$

这是一个关于 $v_t$ 的二次函数。

### 16.3 求解最优 $v_t$

对 $v_t$ 求导：

$$
\begin{aligned}
\frac{\partial\mathcal L}{\partial v_t}
={}&
2A(x_t+Av_t-x_{t+\Delta}-Bv_+)\\
&+
2\eta(v_t-v_+).
\end{aligned}
$$

令梯度为零：

$$
A(x_t+Av_t-x_{t+\Delta}-Bv_+)
+
\eta(v_t-v_+)=0.
$$

整理：

$$
(A^2+\eta)v_t
=
A(x_{t+\Delta}-x_t)
+
(AB+\eta)v_+.
$$

因此：

$$
\boxed{
v_t
=
\frac{A(x_{t+\Delta}-x_t)}
{A^2+\eta}
+
\frac{AB+\eta}
{A^2+\eta}v_+.
}
\tag{16.2}
$$

这就是局部最优预测的表达式。

### 16.4 定义 oracle velocity

论文定义：

$$
\boxed{
x_T=x_t+(T-t)v^*(x_t,t).
}
\tag{16.3}
$$

因此：

$$
\boxed{
v^*(x_t,t)
=
\frac{x_T-x_t}{T-t}.
}
\tag{16.4}
$$

它表示从当前状态到分段终点的平均位移速度。

注意，这个速度不一定等于真实边缘 ODE 的瞬时速度：

$$
u(x_t,t).
$$

只有当真实 ODE 在当前分段内匀速运动时，两者才相等。

### 16.5 推导误差递推

定义：

$$
e_t=v_t-v_t^*,
$$

$$
e_+=v_+-v_+^*.
$$

根据 oracle 定义：

$$
x_T=x_t+Av_t^*,
$$

$$
x_T=x_{t+\Delta}+Bv_+^*.
$$

两式相减：

$$
\boxed{
x_{t+\Delta}-x_t
=
Av_t^*-Bv_+^*.
}
\tag{16.5}
$$

代入式（16.2）：

$$
\begin{aligned}
v_t
={}&
\frac{A(Av_t^*-Bv_+^*)}{A^2+\eta}\\
&+
\frac{AB+\eta}{A^2+\eta}v_+.
\end{aligned}
$$

使用：

$$
v_+=v_+^*+e_+,
$$

得到：

$$
\begin{aligned}
v_t
={}&
\frac{A^2}{A^2+\eta}v_t^*\\
&+
\frac{\eta}{A^2+\eta}v_+^*\\
&+
\frac{AB+\eta}{A^2+\eta}e_+.
\end{aligned}
$$

两边减去：

$$
v_t^*,
$$

得到：

$$
\boxed{
e_t
=
\frac{\eta}{A^2+\eta}
(v_+^*-v_t^*)
+
\frac{AB+\eta}{A^2+\eta}e_+.
}
\tag{16.6}
$$

恢复原始符号：

$$
\boxed{
\begin{aligned}
v_\theta(x_t,t)-v^*(x_t,t)
={}&
\frac{\eta}{(T-t)^2+\eta}
\left[
v^*(x_{t+\Delta},t+\Delta)
-
v^*(x_t,t)
\right]\\
&+
\frac{(T-t)(T-t-\Delta)+\eta}
{(T-t)^2+\eta}\\
&\quad\cdot
\left[
v_{\theta^-}(x_{t+\Delta},t+\Delta)
-
v^*(x_{t+\Delta},t+\Delta)
\right].
\end{aligned}
}
\tag{16.7}
$$

这对应论文 Theorem 2 的误差结构。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

### 16.6 关于分段权重的一个细节

上面的推导暂时忽略了：

$$
\omega_i.
$$

但论文公式（8）实际包含：

$$
\omega_i\|f_t-f_+\|^2.
$$

如果严格保留该权重，则局部损失为：

$$
\mathcal L(v_t)
=
\omega_i\|x_t+Av_t-x_{t+\Delta}-Bv_+\|^2
+
\eta\|v_t-v_+\|^2.
$$

求导后得到：

$$
\boxed{
v_t
=
\frac{
\omega_i A(x_{t+\Delta}-x_t)
+
(\omega_i AB+\eta)v_+
}{
\omega_i A^2+\eta
}.
}
\tag{16.8}
$$

因此，相应的误差递推应为：

$$
\boxed{
e_t
=
\frac{\eta}{\omega_i A^2+\eta}
(v_+^*-v_t^*)
+
\frac{\omega_i AB+\eta}
{\omega_i A^2+\eta}e_+.
}
\tag{16.9}
$$

如果定义有效权重：

$$
\eta_{\mathrm{eff}}=\frac{\eta}{\omega_i},
$$

那么可以将它写成与论文公式（10）相同的形式。

因此，**论文 Theorem 2 的系数实际上对应终点损失权重为 1，或者将分段权重吸收到速度损失系数中的情况。**

这是阅读原文时需要留意的一个细节。

### 16.7 Theorem 2 的含义

误差包含两部分。

第一部分：

$$
\boxed{
\frac{\eta}{A^2+\eta}
(v_+^*-v_t^*)
}
\tag{16.10}
$$

表示 oracle velocity 自身不一致带来的误差。

第二部分：

$$
\boxed{
\frac{AB+\eta}{A^2+\eta}e_+
}
\tag{16.11}
$$

表示下一时刻目标预测误差的传播。

如果真实边缘速度在整个分段内一致：

$$
u(x_t,t)=u(x_{t+\Delta},t+\Delta),
$$

那么 oracle velocity 也一致：

$$
v_t^*=v_+^*.
$$

第一项消失。

如果终点条件正确，并且局部优化能够达到理想解，则误差可以沿时间递推消失。

这就是论文 Corollary 2.1 的核心思想。

但是，如果真实速度场本来就是弯曲的：

$$
v_t^*\neq v_+^*,
$$

第一项通常不会自动消失。

所以：

$$
\boxed{
\text{Theorem 2 没有证明任意真实概率流都能被无误差地直线化。}
}
\tag{16.12}
$$

---

## 17. Corollary 2.1：什么时候能够恢复真实速度场？

论文给出一个特殊情形。

假设真实边缘速度场 $u$ 在当前分段内已经满足速度一致性：

$$
\boxed{
\frac{d}{dt}u(x_t,t)=0,
\qquad
\frac{dx_t}{dt}=u(x_t,t).
}
\tag{17.1}
$$

那么：

$$
u(x_t,t)=u_0.
$$

因此：

$$
x_T=x_t+(T-t)u_0.
$$

根据 oracle velocity 定义：

$$
v^*(x_t,t)=\frac{x_T-x_t}{T-t},
$$

得到：

$$
\boxed{
v^*(x_t,t)=u_0=u(x_t,t).
}
\tag{17.2}
$$

此时：

$$
v_+^*-v_t^*=0.
$$

所以 Theorem 2 的误差递推变成：

$$
\boxed{
e_t
=
\frac{AB+\eta}{A^2+\eta}e_+.
}
\tag{17.3}
$$

如果终点预测误差为零：

$$
e_T=0,
$$

则通过递推：

$$
e_t=0.
$$

因此：

$$
\boxed{
v_\theta(x_t,t)=u(x_t,t).
}
\tag{17.4}
$$

这就是论文 Corollary 2.1 的主要结论。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

**需要强调：这个结论的前提是，真实边缘速度场已经一致。**

它并没有解决一般情况下：

$$
\partial_tu+J_uu\neq0
$$

时，如何同时保证速度一致性和分布匹配的问题。

另外，实际神经网络采用共享参数、EMA 目标和随机梯度优化，不能直接把逐点二次优化的理想结论理解为有限模型训练的严格收敛保证。

---

## 18. 使用 Consistency-FM 蒸馏预训练模型

论文除了直接训练，还提出从预训练 Flow Matching 模型蒸馏。

设教师速度场为：

$$
u_\phi(x,t).
$$

教师定义 ODE：

$$
\frac{dx_t}{dt}=u_\phi(x_t,t).
$$

给定当前状态：

$$
x_t,
$$

使用教师进行一次 Euler 更新：

$$
\boxed{
\hat x_{t+\Delta}^{\phi}
=
x_t+\Delta u_\phi(x_t,t).
}
\tag{18.1}
$$

然后使用学生模型：

$$
v_\theta
$$

和 EMA 学生：

$$
v_{\theta^-}
$$

构造一致性损失：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{distill}}
=
\mathbb E\Big[
&
\|f_\theta(x_t,t)
-
f_{\theta^-}(\hat x_{t+\Delta}^{\phi},t+\Delta)\|^2\\
&+
\eta
\|v_\theta(x_t,t)
-
v_{\theta^-}(\hat x_{t+\Delta}^{\phi},t+\Delta)\|^2
\Big].
\end{aligned}
}
\tag{18.2}
$$

这是论文公式（11）。[arXiv](https://arxiv.org/html/2407.02398?utm_source=chatgpt.com)

与直接使用预定义条件插值不同，这里：

$$
\hat x_{t+\Delta}^{\phi}
$$

来自教师 ODE 的局部推进。

因此，它与教师生成轨迹的联系更加直接。

但教师速度场本身可能是弯曲的，而学生仍被要求提高速度一致性。

所以这是一种对教师动力学的近似压缩，而不是严格保证学生完全复现教师的全部概率流。

此外，论文正文给出了蒸馏目标，但其主要实验集中在从头训练的无条件图像生成任务，不能把蒸馏方案的效果视为已经获得同等充分的实验验证。

---

## 19. 推理：单步与分段 Euler

### 19.1 单段采样

如果：

$$
K=1,
$$

初始采样：

$$
x_0\sim p_0.
$$

预测速度：

$$
v_\theta(x_0,0).
$$

一步更新：

$$
\boxed{
\hat x_1=x_0+v_\theta(x_0,0).
}
\tag{19.1}
$$

如果速度沿模型 ODE 轨迹严格一致，那么这一步可以精确得到 ODE 终点。

但实际模型通常只能近似满足一致性，因此仍然存在误差。

### 19.2 分段采样

将时间划分为：

$$
t_i=\frac{i}{K},
\qquad i=0,\ldots,K.
$$

每段使用：

$$
\boxed{
x_{t_{i+1}}
=
x_{t_i}
+
\frac1K v_\theta^i(x_{t_i},t_i).
}
\tag{19.2}
$$

因此：

- $K=1$：1 NFE；
- $K=2$：2 NFE；
- $K=4$：4 NFE。

也可以在每段内部使用多个 Euler 步。

需要注意，原文公式（12）的索引写法存在不够严谨之处；这里使用从 $i=0$ 到 $K-1$ 的完整递推，明确包含最后一段。

### 19.3 分段的实际意义

从数值分析角度，Euler 更新为：

$$
x_{t+h}
=
x_t+h\,v_\theta(x_t,t).
$$

对于一般速度场，局部截断误差与沿轨迹的速度变化有关。

对真实 ODE 解作 Taylor 展开：

$$
x_{t+h}
=
x_t+h\dot x_t+\frac{h^2}{2}\ddot x_t+O(h^3).
$$

因为：

$$
\dot x_t=v_\theta(x_t,t),
$$

所以：

$$
\ddot x_t
=
\partial_tv_\theta+J_{v_\theta}v_\theta.
$$

因此：

$$
\boxed{
\begin{aligned}
x_{t+h}
={}&
x_t+h\,v_\theta(x_t,t)\\
&+
\frac{h^2}{2}
\left(
\partial_tv_\theta+J_{v_\theta}v_\theta
\right)
+
O(h^3).
\end{aligned}
}
\tag{19.3}
$$

Euler 方法忽略了第二项。

因此其局部截断误差为：

$$
\boxed{
e_{\mathrm{Euler}}
=
\frac{h^2}{2}
\left(
\partial_tv_\theta+J_{v_\theta}v_\theta
\right)
+
O(h^3).
}
\tag{19.4}
$$

这个公式直接解释了 Consistency-FM 的数值分析动机：

如果能够让：

$$
\partial_tv_\theta+J_{v_\theta}v_\theta
$$

接近零，那么 Euler 的局部截断误差也会减小。

所以速度一致性确实可能改善少步 Euler 采样。

但这里仍然存在两个独立误差：

$$
\boxed{
\text{总生成误差}
=
\text{速度场学习误差}
+
\text{数值积分误差的影响}
}
\tag{19.5}
$$

这只是概念性分解，并非两项误差可以直接线性相加。

**Consistency-FM 主要希望减小第二类误差，但如果速度场偏离正确的概率输运动力学，那么积分得再准确，也不能自动保证生成分布正确。**

---

# 第五部分：理论审查与最终结论

## 20. 速度一致性是否保证原始边际概率路径？

设原始 Flow Matching 的边缘速度为：

$$
u(x,t).
$$

它满足：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tu)=0.
}
\tag{20.1}
$$

Consistency-FM 学习到：

$$
v_\theta=u+\delta v.
$$

如果要求它生成相同的边际概率路径：

$$
p_t,
$$

则必须满足：

$$
\partial_tp_t+\nabla\cdot(p_tv_\theta)=0.
$$

代入：

$$
v_\theta=u+\delta v,
$$

得到：

$$
\partial_tp_t+\nabla\cdot(p_tu)
+
\nabla\cdot(p_t\delta v)=0.
$$

由于原始连续性方程成立：

$$
\partial_tp_t+\nabla\cdot(p_tu)=0,
$$

因此：

$$
\boxed{
\nabla\cdot(p_t\delta v)=0.
}
\tag{20.2}
$$

这说明，新的速度场可以不同于原始边缘速度，但要保持原始概率路径，其差异必须满足相应的概率流守恒条件。

Consistency-FM 的损失并没有直接保证式（20.2）。

所以：

$$
\boxed{
\text{速度一致性}
\not\Rightarrow
\text{原始边际概率路径保持}.
}
\tag{20.3}
$$

不过，如果模型学习到另一条概率路径：

$$
\tilde p_t,
$$

并且：

$$
\tilde p_0=p_0,
\qquad
\tilde p_1=p_1,
$$

同时满足：

$$
\partial_t\tilde p_t+
\nabla\cdot(\tilde p_tv_\theta)=0,
$$

那么仍然可以正确生成目标分布。

因此，我们不能简单认为偏离原始 $u$ 就必然错误。

真正缺少的是：**论文没有对一般情况下的新速度场给出充分的终点分布正确性保证。**

## 21. 速度一致性是否保证最优传输？

最优传输要求：

$$
\boxed{
\min_{\pi\in\Pi(p_0,p_1)}
\mathbb E_\pi\|X_1-X_0\|^2.
}
\tag{21.1}
$$

在动态形式下，二次成本最优传输可以写为：

$$
\boxed{
\min_{p_t,v_t}
\int_0^1
\int_{\mathbb R^d}
p_t(x)\|v_t(x)\|^2\,dx\,dt
}
\tag{21.2}
$$

约束为：

$$
\boxed{
\partial_tp_t+\nabla\cdot(p_tv_t)=0,
}
\tag{21.3}
$$

以及：

$$
p_{t=0}=p_0,
\qquad
p_{t=1}=p_1.
$$

这是 Benamou–Brenier 动态最优传输问题。

它同时包含：

1. 概率质量守恒；
2. 初始和终点分布约束；
3. 动能成本最小化。

而 Consistency-FM 的主要动力学约束为：

$$
\boxed{
\partial_tv+J_vv=0.
}
\tag{21.4}
$$

这个条件并没有直接包含输运成本：

$$
\int_0^1\int p_t\|v_t\|^2\,dx\,dt.
$$

因此：

$$
\boxed{
\text{速度一致性}
\not\Rightarrow
\text{二次成本最优传输}.
}
\tag{21.5}
$$

虽然二次成本最优传输的位移插值具有直线、匀速的特点，但反过来，任意直线输运不一定具有最优性。

## 22. 论文的理论贡献与局限

综合前面的推导，可以将论文的理论贡献分成两个层次。

**已经建立的数学联系：**

$$
\boxed{
\text{沿自身 ODE 的速度一致}
\Longleftrightarrow
\text{自身 ODE 轨迹匀速直线}
}
$$

以及：

$$
\boxed{
\text{速度变化率越小}
\Longrightarrow
\text{Euler 局部截断误差的主导项越小}
}
$$

第二个结论需要速度场足够光滑，并且讨论的是局部数值积分误差。

**尚未获得一般性证明的部分：**

$$
\boxed{
\begin{gathered}
\text{给定任意训练 coupling}\\
\Downarrow\\
\text{Consistency-FM 训练}\\
\Downarrow\\
\text{生成 ODE 严格直线}\\
\Downarrow\\
\text{正确匹配目标分布}\\
\Downarrow\\
\text{达到最优传输}
\end{gathered}
}
$$

尤其需要注意：

- 训练条件插值路径与模型自身 ODE 轨迹并不相同；
- 原始边缘速度是条件速度的加权期望；
- 一致性损失可能改变原始边缘速度场；
- 改变速度场后，原始连续性方程不一定继续成立；
- 直线性不等于最优传输；
- 分段训练改善局部积分性质，不自动保证终点分布正确。

这些是我们基于论文公式和 Flow Matching 理论得到的分析，不应全部表述成作者已经承认或证明的结论。

---

# 六、最终总结

这篇论文的研究思路可以概括为：

**标准 Flow Matching 的条件训练路径虽然是匀速直线，但由于 MSE 学习的是条件速度的加权期望，最终生成 ODE 仍可能弯曲。Consistency-FM 希望通过终点一致性和速度一致性约束，使模型学习到更容易进行少步 Euler 积分的速度场。**

其核心训练目标为：

$$
\boxed{
\begin{aligned}
\mathcal L_{\mathrm{CFM}}
=
\mathbb E\Big[
&
\|f_\theta(X_t,t)
-
f_{\theta^-}(X_{t+\Delta},t+\Delta)\|^2\\
&+
\eta
\|v_\theta(X_t,t)
-
v_{\theta^-}(X_{t+\Delta},t+\Delta)\|^2
\Big],
\end{aligned}
}
$$

其中：

$$
\boxed{
f_\theta(x,t)=x+(1-t)v_\theta(x,t).
}
$$

它的主要价值在于：**把生成 ODE 的速度变化直接作为训练阶段需要考虑的因素，而不是完全依赖推理时增加积分步数来改善采样质量。**

但我们的核心判断是：

$$
\boxed{
\text{少步采样有效}
\neq
\text{理论上保证正确的直线输运}
}
$$

论文实验支持该方法在特定图像生成任务上的有效性，但其理论结果不足以证明一般情况下的概率分布保持或最优传输。

因此，这篇论文适合作为**速度场正则化与少步生成方法**来学习，而不宜将其作为 Flow Matching 最优输运理论的基础性结论。

**与 Rectified Flow 的区别尤其值得记住：**

- **Rectified Flow / Reflow：** 通过重新构造生成配对，改善 coupling 和轨迹结构。
- **Consistency-FM：** 不显式优化 coupling，而是通过一致性损失约束学习到的速度场。

前者更直接地处理输运耦合，后者更直接地处理速度场的可积分性。

这也是两条研究路线最根本的区别。

**主要参考：** [Consistency Flow Matching 原文及附录](https://arxiv.org/html/2407.02398)、[Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747)、[Flow Straight and Fast: Rectified Flow](https://arxiv.org/abs/2209.03003)、[Consistency Models](https://arxiv.org/abs/2303.01469)。